In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import arxiv
import requests
import tarfile
import tiktoken
import numpy as np
from pptx.dml.color import RGBColor
from pptx.oxml.xmlchemy import OxmlElement
from pdf2image import convert_from_path
from pptx import Presentation
from pptx.util import Pt, Inches
import re
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

PAPER_ID = "2502.21104"  # Edit this with the paper id you want to make a ppt of

TITLE_FONT = "Calibri"
BODY_FONT = "Segoe UI"
POINT_FONT = "Segoe UI"
ACCENT = RGBColor(27, 6, 84)
DARK = RGBColor(40, 40, 40) 
MUTED = RGBColor(90, 90, 90)
TITLE_SIZE = Pt(26)

In [33]:
load_dotenv()

True

In [34]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("KEY"),
)

DATA_DIR = f"./tex_sources/{PAPER_ID}"
VECTOR_FILE = "vectors.npy"
DOC_FILE = "docs.json"

In [35]:
def answer(messages, reasoning: bool=False):
    response = client.chat.completions.create(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    messages=messages,
    extra_body={"reasoning": {"enabled": reasoning}}
    )

    return response.choices[0].message

In [36]:
def search_arxiv(query, max_results=5):
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    client = arxiv.Client()
    results = []
    for paper in client.results(search):
        results.append({
            "title": paper.title,
            "authors": [a.name for a in paper.authors],
            "summary": paper.summary,
            "url": paper.entry_id,
            "published": str(paper.published.date())
        })

    return results

In [37]:
def download_tex_source(arxiv_id, save_path):
    os.makedirs(save_path, exist_ok=True)
    res = requests.get(f"https://arxiv.org/src/{arxiv_id}")
    if res.status_code != 200:
        print(f"Failed to download TeX source for {arxiv_id}. Status code: {res.status_code}")
        return
    with open(os.path.join(save_path, f"{arxiv_id}.tar.gz"), "wb") as f:
        f.write(res.content)
    with tarfile.open(os.path.join(save_path, f"{arxiv_id}.tar.gz")) as tar:
        os.makedirs(os.path.join(save_path, arxiv_id), exist_ok=True)
        tar.extractall(path=os.path.join(save_path, arxiv_id))

    print(f"Downloaded & Extracted TeX source for {arxiv_id} to {save_path}/{arxiv_id}")

In [38]:
download_tex_source(PAPER_ID, "./tex_sources")

Downloaded & Extracted TeX source for 2602.02710 to ./tex_sources/2602.02710


C:\Users\Divyansh\AppData\Local\Temp\ipykernel_24228\2200843505.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=os.path.join(save_path, arxiv_id))


In [39]:
SYSTEM_PROMPT_SUMMARIZER = """
You are an AI research assistant.

You are working with a bunch of other agents, your goal in this ecosystem is to help with summarizing content from tex sources of a paper.

You will need to read through the tex source files from a given directory. You should first aim to find the main tex file (usually the one that includes others or has the title/author/abstract sections). You may need to read through multiple files if the paper is split across several tex files but finding the main tex file is crucial to understand the structure of the paper.

You must summarize with the goal in mind that the summary will help in creating a powerpoint presentation of the paper.
Your summary must clearly give topics, subtopics, and key points and/or paragraphs for each slide, yet should cover all major sections of the paper with good detail. You need not concern yourself with adding diagrams/images, however, you need to provide suggestions for diagrams/images only where relevant.
The layout of each slide is not your concern, just the textual content of each slide in a hierarchical way.

Your output should be EXACTLY in this format:

Name: The exact title of the paper

Citation: The exact citation of the paper in this format - "Authors. Title. Venue, Year. URL"

Slide 1: Title of Slide (Preferably the "abstract" or "introduction" for the first slide)
- Subtitle of Slide: [if any]
- Paragraph/Sentence 1: ...
- Paragraph/Sentence 2: ...
- ...
- Key Point 1: ...
- Key Point 2: ...
- Paragraph/Sentence 3: ...
- ...
- Diagram/Image Suggestion: [if any]

Slide 2: Title of Slide
- ...

- ...

After you provide the summary, your work will be evaluated by another agent for comprehensiveness.
You may need to iterate based on their feedback in the same format.
"""

In [40]:
def chunk_text(text, max_tokens=900):
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)

    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk = enc.decode(tokens[i : i + max_tokens])
        chunks.append(chunk)

    return chunks

In [41]:
def build_vector_db():
    docs = []
    meta = []

    for file in os.listdir(DATA_DIR):
        if not file.endswith(".tex"):
            continue

        path = os.path.join(DATA_DIR, file)

        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        chunks = chunk_text(text)

        for i, c in enumerate(chunks):
            docs.append(c)
            meta.append({"file": file, "chunk": i})

    vectors = []

    for d in docs:
        emb = (
            client.embeddings.create(model="text-embedding-3-small", input=d)
            .data[0]
            .embedding
        )

        vectors.append(emb)

    vectors = np.array(vectors, dtype=np.float32)

    # save
    np.save(VECTOR_FILE, vectors)

    with open(DOC_FILE, "w", encoding="utf-8") as f:
        json.dump({"docs": docs, "meta": meta}, f)

    print("Vector DB built:", vectors.shape)

In [42]:
build_vector_db()

Vector DB built: (49, 1536)


In [43]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def retrieve(query, k=8):

    vectors = np.load(VECTOR_FILE)

    with open(DOC_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    q_emb = (
        client.embeddings.create(model="text-embedding-3-small", input=query)
        .data[0]
        .embedding
    )

    scores = [cosine(q_emb, v) for v in vectors]

    idx = np.argsort(scores)[::-1][:k]

    retrieved = [data["docs"][i] for i in idx]

    return "\n\n".join(retrieved)

In [44]:
def summarizer(reasoning=True):
    query = """
    Find content about title, abstract, introduction,
    method, experiments, results, conclusion and key equations.
    """

    retrieved = retrieve(query)

    prompt = f"""
Use the retrieved LaTeX content below to create the summary.

Retrieved Content:
------------------
{retrieved}
------------------

Follow the required slide format EXACTLY.
"""

    context = [
        {"role": "system", "content": SYSTEM_PROMPT_SUMMARIZER},
        {"role": "user", "content": prompt},
    ]

    response = client.chat.completions.create(
        model="stepfun/step-3.5-flash:free",
        messages=[{"role": "system", "content": SYSTEM_PROMPT_SUMMARIZER}],
        extra_body={"reasoning": {"enabled": reasoning}},
    )

    prompt = "Use the uploaded LaTeX source documents available in your knowledge base. Please summarize the content from the tex source files in the specified format."
    try:
        context.append(
                {
                    "role": "user",
                    "content": prompt
                }
            )
        response = answer(context, reasoning)
        res = response.content
        print(res)
        return res
    except Exception as e:
        print(e)

In [45]:
res = summarizer(reasoning=True)

Name: Maximum Likelihood Reinforcement Learning  
Citation: Fahim Tajwar, Guanning Zeng, Yueer Zhou, Yuda Song, Daman Arora, Yiding Jiang, Jeff Schneider, Ruslan Salakhutdinov, Haiwen Feng, Andrea Zanette. Maximum Likelihood Reinforcement Learning. ICLR 2025. https://arxiv.org/abs/xxxx  Slide 1: Maximum Likelihood Reinforcement Learning  
- Subtitle of Slide: Abstract  
- Paragraph/Sentence 1: We introduce Maximum Likelihood Reinforcement Learning (MLRL), a framework that trains large language models using a likelihood‑maximization objective instead of traditional reward modeling.  
- Paragraph/Sentence 2: MLRL replaces the reward model with a direct policy objective that evaluates entire response sequences.  
- Paragraph/Sentence 3: Experiments on three benchmark suites—reasoning, instruction following, and maze solving—show that MLRL matches or exceeds state‑of‑the‑art baselines such as GRPO and REINFORCE.  
- Key Point 1: Simpler training pipeline without a separate reward model.  


In [46]:
SYSTEM_PROMPT_PPTMAKER = """
You format slides AND select appropriate images.

You MUST extract these top-level fields:

- Name → field "name"
- Citation → field "citation"

You receive:
1. Slide summary
2. List of available images with filenames.

For each slide:
- Use the diagram suggestion text to pick the BEST matching image path.
- If no match exists, set image_path: null.
- Do NOT invent new text.

Output JSON:

{
  "name": "...",
  "citation": "...",
  "slides":[
    {
      "title":"",
      "paragraphs":[],
      "key_points":[],
      "diagram":"",
      "image_path":"path/to/file.png or null"
    }
  ]
}
"""

In [47]:
def gather_images_recursive(root_dir):
    exts = (".png", ".jpg", ".jpeg", ".pdf", ".eps")

    images = []

    for dirpath, _, filenames in os.walk(root_dir):

        for f in filenames:
            if f.lower().endswith(exts):

                full = os.path.join(dirpath, f)

                images.append(
                    {
                        "path": full,
                        "name": f,
                        "folder": os.path.relpath(dirpath, root_dir),
                    }
                )

    return images

In [48]:
images = gather_images_recursive(f"./tex_sources/{PAPER_ID}")
print(len(images))

19


In [49]:
def find_referenced_images(root_dir):

    pattern = re.compile(r"includegraphics(?:\[.*?\])?\{([^}]+)\}")

    referenced = set()

    for dirpath, _, files in os.walk(root_dir):

        for f in files:
            if f.endswith(".tex"):
                text = open(
                    os.path.join(dirpath, f), encoding="utf-8", errors="ignore"
                ).read()

                for match in pattern.findall(text):
                    referenced.add(match)

    return referenced

In [50]:
def gather_smart_images(root_dir):

    all_imgs = gather_images_recursive(root_dir)
    refs = find_referenced_images(root_dir)

    for img in all_imgs:

        name_no_ext = os.path.splitext(img["name"])[0]

        img["likely_used"] = img["name"] in refs or name_no_ext in refs

    return all_imgs

In [51]:
images = gather_smart_images(f"./tex_sources/{PAPER_ID}")

In [52]:
def parse_summary(summary_text):
    slides = []

    slide_blocks = re.split(r"Slide \d+:", summary_text)[1:]

    for block in slide_blocks:
        lines = block.strip().split("\n")

        title = lines[0].strip()

        bullets = []
        diagram = None

        for line in lines[1:]:
            line = line.strip()

            if not line:
                continue

            if line.startswith("- Diagram/Image Suggestion:"):
                diagram = line.replace("- Diagram/Image Suggestion:", "").strip()

            elif line.startswith("-"):
                bullets.append(line[1:].strip())

        slides.append({
            "title": title,
            "bullets": bullets,
            "diagram": diagram
        })

    return slides


In [53]:
def extract_json(text):

    text = text.strip()

    if "```" in text:
        matches = re.findall(r"```(?:json)?(.*?)```", text, re.DOTALL)
        if matches:
            text = matches[0]

    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        text = m.group(0)

    text = re.sub(r",\s*([\]}])", r"\1", text)

    return json.loads(text)

In [54]:
def ensure_image(path):
    """
    Returns a real image path suitable for python-pptx.
    Converts PDFs/EPS to PNG if needed.
    """

    if not path or not os.path.exists(path):
        return None

    ext = path.lower().split(".")[-1]

    if ext in ["png", "jpg", "jpeg", "bmp", "gif", "tiff"]:
        return path

    if ext == "pdf":
        try:
            pages = convert_from_path(path, first_page=1, last_page=1, dpi=150)
            out = path + ".png"
            pages[0].save(out, "PNG")
            return out
        except Exception as e:
            print("PDF convert failed:", path, e)
            return None

    if ext == "eps":
        try:
            img = Image.open(path)
            out = path + ".png"
            img.save(out, "PNG")
            return out
        except Exception:
            return None

    return None

In [55]:
def generate_ppt_from_summary(summary, reasoning=True):
    context = [
        {"role": "system", "content": SYSTEM_PROMPT_PPTMAKER},
        {
            "role": "user",
            "content": f"""
Slides Summary:
{summary}

Available Images:
{json.dumps(images, indent=2)}
""",
        },
    ]

    response = client.chat.completions.create(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        messages=context,
        extra_body={"reasoning": {"enabled": reasoning}},
    )

    try:
        structured = extract_json(response.choices[0].message.content)
        if "name" not in structured:
            structured["name"] = structured["slides"][0]["title"]
        if "citation" not in structured:
            structured["citation"] = ""
    except Exception as e:
        print("Invalid JSON from agent")
        print(response.choices[0].message.content)
        return

    prs = Presentation()

    cover = prs.slides.add_slide(prs.slide_layouts[0])

    cover_title = cover.shapes.title.text_frame.paragraphs[0]
    cover_title.text = structured["slides"][0]["title"]
    cover_title.font.name = TITLE_FONT
    cover_title.font.size = Pt(36)
    cover_title.font.bold = True

    sub = cover.placeholders[1].text_frame.paragraphs[0]
    sub.text = "Research Paper Presentation"
    sub.font.name = BODY_FONT
    sub.font.size = Pt(20)

    for s in structured["slides"]:

        slide = prs.slides.add_slide(prs.slide_layouts[1])
        slide.shapes.title.text = s["title"]

        body = slide.shapes.placeholders[1].text_frame
        body.clear()

        for i, p_text in enumerate(s.get("paragraphs", [])):

            p = body.paragraphs[0] if i == 0 else body.add_paragraph()
            p.text = p_text

            p.font.name = BODY_FONT
            p.font.size = Pt(18)
            p.font.color.rgb = DARK

            pPr = p._p.get_or_add_pPr()
            pPr.insert(0, OxmlElement('a:buNone'))

        for kp in s.get("key_points", []):

            p = body.add_paragraph()
            p.text = kp

            p.font.name = POINT_FONT
            p.font.size = Pt(18)
            p.font.bold = True
            p.font.color.rgb = DARK

            pPr = p._p.get_or_add_pPr()
            pPr.insert(0, OxmlElement('a:buChar'))

        if s.get("image_path") and os.path.exists(s["image_path"]):
            left = Inches(7)
            top = Inches(2)
            img = ensure_image(s.get("image_path"))
            if img:
                try:
                    slide.shapes.add_picture(img, left, top, width=Inches(3))
                except Exception as e:
                    print("Skipping image:", img, e)

    return structured

In [56]:
def make_cover(prs, title_text):
    cover = prs.slides.add_slide(prs.slide_layouts[0])

    t = cover.shapes.title.text_frame.paragraphs[0]
    t.text = title_text
    t.font.name = TITLE_FONT
    t.font.size = Pt(36)
    t.font.bold = True
    t.font.color.rgb = ACCENT

    s = cover.placeholders[1].text_frame.paragraphs[0]
    s.text = "Research Paper Presentation"
    s.font.name = BODY_FONT
    s.font.size = Pt(20)

    return cover

In [57]:
def add_footer(slide, citation):

    if not citation:
        return

    left = Inches(0.5)
    top = Inches(6.9)

    box = slide.shapes.add_textbox(left, top, Inches(9), Inches(0.5))
    p = box.text_frame.paragraphs[0]

    p.text = citation
    p.font.name = BODY_FONT
    p.font.size = Pt(10)
    p.font.italic = True
    p.font.color.rgb = MUTED

In [58]:
def write_text(slide, s):

    body = slide.shapes.placeholders[1].text_frame
    body.clear()

    for i, p_text in enumerate(s.get("paragraphs", [])):
        p = body.paragraphs[0] if i == 0 else body.add_paragraph()
        p.text = p_text
        p.font.name = BODY_FONT
        p.font.size = Pt(16)
        p.font.color.rgb = DARK
        p.level = 0
        p.line_spacing = 1.1
        p.space_after = Pt(2)

        pPr = p._p.get_or_add_pPr()
        pPr.insert(0, OxmlElement("a:buNone"))
        pPr.set("marL", "0")
        pPr.set("indent", "0")  

    for kp in s.get("key_points", []):
        p = body.add_paragraph()
        p.text = kp
        p.font.name = POINT_FONT
        p.font.size = Pt(16)
        p.font.bold = True
        p.line_spacing = 1.0
        p.space_after = Pt(0)
        p.space_before = Pt(0) 

        pPr = p._p.get_or_add_pPr()
        pPr.insert(0, OxmlElement("a:buChar"))

In [59]:
def render_beautiful_ppt(layout_data, output="final.pptx"):
    if not layout_data:
        raise ValueError("layout_data is None or empty")

    slides = layout_data.get("slides", [])

    if not slides:
        raise ValueError("No slides returned by layout agent")

    prs = Presentation()

    paper_title = layout_data.get("name") or layout_data["slides"][0]["title"]
    cover = make_cover(prs, paper_title)

    citation = layout_data.get("citation")
    add_footer(cover, citation)

    for s in layout_data["slides"]:

        layout = s.get("layout", "text_only")

        if layout == "text_only":
            slide = prs.slides.add_slide(prs.slide_layouts[1])
            slide.shapes.title.text_frame.word_wrap = True
            t = slide.shapes.title.text_frame.paragraphs[0]
            t.text = s["title"]
            t.font.name = TITLE_FONT
            t.font.size = TITLE_SIZE
            t.font.color.rgb = ACCENT

            write_text(slide, s)
            add_footer(slide, citation)

        elif layout == "image_right":
            slide = prs.slides.add_slide(prs.slide_layouts[1])
            slide.shapes.title.text_frame.word_wrap = True
            t = slide.shapes.title.text_frame.paragraphs[0]
            t.text = s["title"]
            t.font.name = TITLE_FONT
            t.font.size = TITLE_SIZE
            t.font.color.rgb = ACCENT

            write_text(slide, s)

            img = ensure_image(s.get("image_path"))
            if img:
                slide.shapes.add_picture(img, Inches(7), Inches(2), width=Inches(3))

            add_footer(slide, citation)

        elif layout == "image_full":
            slide = prs.slides.add_slide(prs.slide_layouts[5])
            slide.shapes.title.text_frame.word_wrap = True
            t = slide.shapes.title.text_frame.paragraphs[0]
            t.text = s["title"]
            t.font.name = TITLE_FONT
            t.font.size = TITLE_SIZE
            t.font.color.rgb = ACCENT

            img = ensure_image(s.get("image_path"))
            if img:
                slide.shapes.add_picture(img, Inches(1), Inches(1.5), width=Inches(8))

            add_footer(slide, citation)

        elif layout == "two_part":
            slide1 = prs.slides.add_slide(prs.slide_layouts[1])
            t = slide1.shapes.title.text_frame.paragraphs[0]
            t.text = s["title"]
            t.font.name = TITLE_FONT
            t.font.size = TITLE_SIZE
            t.font.color.rgb = ACCENT

            write_text(slide1, s)
            add_footer(slide1, citation)

            slide2 = prs.slides.add_slide(prs.slide_layouts[5])
            t = slide2.shapes.title.text_frame.paragraphs[0]
            t.text = s["title"]
            t.font.name = TITLE_FONT
            t.font.size = TITLE_SIZE
            t.font.color.rgb = ACCENT

            img = ensure_image(s.get("image_path"))
            if img:
                left = Inches(6.8)
                top = Inches(1.6)
                slide2.shapes.add_picture(img, left, top, width=Inches(5.2))

            add_footer(slide2, citation)

    prs.save(output)

In [60]:
def simple_layout_agent(structured):

    new_slides = []

    for s in structured.get("slides", []):

        kp = s.get("key_points", [])
        paras = s.get("paragraphs", [])
        img = s.get("image_path")

        if img and (len(kp) > 4 or len(paras) > 2):

            new_slides.append({**s, "image_path": None, "layout": "text_only"})

            new_slides.append(
                {
                    "title": s["title"] + " – Figure",
                    "paragraphs": [],
                    "key_points": [],
                    "image_path": img,
                    "layout": "image_full",
                }
            )

        elif len(kp) > 6:

            mid = len(kp) // 2

            new_slides.append({**s, "key_points": kp[:mid], "layout": "text_only"})

            new_slides.append({**s, "key_points": kp[mid:], "layout": "text_only"})

        else:
            s["layout"] = "image_right" if img else "text_only"
            new_slides.append(s)

    return {
        "name": structured.get("name"),
        "citation": structured.get("citation"),
        "slides": new_slides,
    }

In [61]:
structured = generate_ppt_from_summary(res)
layout_ready = simple_layout_agent(structured)
render_beautiful_ppt(layout_ready)